# 17. Multi-metric Consensus Ranking & Pareto Optimisation

Combine multiple MD analysis metrics (RMSD, H-bond occupancy, QED, RMSF, …)
into a single consensus ranking using weighted Borda count or Z-score normalisation,
and identify the Pareto-optimal compounds.

In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
import pandas as pd

# Metrics DataFrame: rows = sample names, columns = metric names
# Replace with your actual data (e.g. loaded from CSV or built from
# HBondResult.summary, RMSDResult.df, ADMETResult, etc.)
metrics_df = pd.DataFrame(
    {
        "hbond_occ":  [0.85, 0.60, 0.92, 0.45, 0.78],
        "ligand_rmsd": [1.2,  2.5,  0.9,  3.1,  1.8],
        "qed":         [0.72, 0.55, 0.80, 0.48, 0.65],
        "sa_score":    [2.1,  3.4,  1.8,  4.2,  2.9],
    },
    index=["cpd_A", "cpd_B", "cpd_C", "cpd_D", "cpd_E"],
)

# Which metrics are better when higher (True) or lower (False)?
HIGHER_IS_BETTER = {
    "hbond_occ":   True,
    "ligand_rmsd": False,
    "qed":         True,
    "sa_score":    False,
}

# Optional: per-metric weights (default = 1.0 for all)
WEIGHTS = {
    "hbond_occ":   2.0,   # H-bond occupancy weighted more heavily
    "ligand_rmsd": 1.5,
    "qed":         1.0,
    "sa_score":    1.0,
}

# Ranking method: 'borda' or 'zscore'
RANKING_METHOD = "borda"

# Pareto objectives (subset of columns)
PARETO_OBJECTIVES = ["hbond_occ", "ligand_rmsd"]
# ============================================================

In [ ]:
from mdatools.scoring.consensus import ConsensusRanker

ranker = ConsensusRanker(weights=WEIGHTS, higher_is_better=HIGHER_IS_BETTER)
result = ranker.combine(metrics_df, method=RANKING_METHOD, pareto_objectives=PARETO_OBJECTIVES)

print(f"Pareto front ({len(result.pareto_front)} compounds): {result.pareto_front}")
print()
print(result.ranking[["consensus_score", "rank"]])

## Consensus Ranking Chart

In [ ]:
import matplotlib.pyplot as plt
from mdatools.plotting import plot_consensus_ranking

fig = plot_consensus_ranking(result, top_n=10)
plt.show()

## Pareto Front Scatter

In [ ]:
from mdatools.plotting import plot_pareto_front

fig = plot_pareto_front(
    metrics_df,
    x_metric=PARETO_OBJECTIVES[0],
    y_metric=PARETO_OBJECTIVES[1],
    pareto_names=result.pareto_front,
)
plt.show()

## Metric Correlation Heatmap

In [ ]:
from mdatools.plotting import plot_metric_correlation

fig = plot_metric_correlation(metrics_df)
plt.show()